# 100 Days of AI — All-in-One Render & Publish

Renders a lesson **entirely on Colab** (slides + audio + talking head + final composite), uploads it to **YouTube (unlisted)**, and **pushes the updated site to GitHub** — so your live GitHub Pages course updates with a single call. Your Mac does nothing.

**Runtime → Change runtime type → T4 GPU.**

- **Setup (once per session):** run Section 1 (cells 1–5).
- **Per lesson / per day:** Section 2 — `render_and_publish(day, lesson)`.

## Section 1 — Setup (run once per session)

In [ ]:
# Cell 1 — GPU check
import subprocess
r = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
if r.returncode != 0:
    raise RuntimeError('No GPU. Runtime -> Change runtime type -> T4 GPU.')
print('GPU OK')

In [ ]:
# Cell 2 — Wav2Lip: clone + deps + patches + weights (talking-head engine)
import os
!pip install -q "numpy<2.0"
if not os.path.exists('/content/Wav2Lip'):
    !git clone -q https://github.com/justinjohn0306/Wav2Lip.git /content/Wav2Lip
%cd /content/Wav2Lip
!pip install -q "numpy<2.0" librosa==0.10.2.post1 opencv-python numba tqdm batch-face
# compatibility patches for modern Colab (librosa + numpy)
!sed -i 's/librosa.filters.mel(hp.sample_rate, hp.n_fft,/librosa.filters.mel(sr=hp.sample_rate, n_fft=hp.n_fft,/' audio.py
!sed -i 's/librosa\.core\./librosa./g' audio.py
!find . -name '*.py' -exec sed -i 's/np\.float\b/float/g; s/np\.int\b/int/g; s/np\.bool\b/bool/g' {} \;
!mkdir -p checkpoints face_detection/detection/sfd
if not os.path.exists('checkpoints/wav2lip_gan.pth'):
    !wget -q --show-progress 'https://github.com/justinjohn0306/Wav2Lip/releases/download/models/wav2lip_gan.pth' -O checkpoints/wav2lip_gan.pth
if not os.path.exists('checkpoints/mobilenet.pth'):
    !wget -q 'https://github.com/justinjohn0306/Wav2Lip/releases/download/models/mobilenet.pth' -O checkpoints/mobilenet.pth
if not os.path.exists('face_detection/detection/sfd/s3fd.pth'):
    !wget -q 'https://github.com/justinjohn0306/Wav2Lip/releases/download/models/s3fd.pth' -O face_detection/detection/sfd/s3fd.pth
print('Wav2Lip ready.')

In [ ]:
# Cell 3 — Clone the course repo + install pipeline deps
import os
REPO_URL = 'https://github.com/Katiehey/100-Days-of-AI---The-Complete-AI-Engineering-Bootcamp.git'
REPO = '/content/course'
if not os.path.exists(REPO):
    !git clone -q "$REPO_URL" "$REPO"
else:
    !git -C "$REPO" pull -q
%cd $REPO
!pip install -q edge-tts pyyaml pillow requests   # ffmpeg is preinstalled on Colab
print('Course repo ready at', REPO)

In [ ]:
# Cell 4 — Mount Drive + avatar (portrait for the talking head)
from google.colab import drive
drive.mount('/content/drive')
import os, shutil
DRIVE = '/content/drive/MyDrive/100DaysOfAI'
os.makedirs(f'{DRIVE}/videos', exist_ok=True)
os.makedirs(f'{DRIVE}/final', exist_ok=True)
AV_DRIVE, AV_LOCAL = f'{DRIVE}/Avatar.png', '/content/avatar.png'
if os.path.exists(AV_DRIVE):
    shutil.copy(AV_DRIVE, AV_LOCAL); print('Avatar loaded from Drive.')
else:
    from google.colab import files
    print('Upload Avatar.png (saved to Drive for next time):')
    up = files.upload(); name = list(up.keys())[0]
    shutil.move(name, AV_LOCAL); shutil.copy(AV_LOCAL, AV_DRIVE)
from IPython.display import Image, display
display(Image(AV_LOCAL, width=180))

In [ ]:
# Cell 5 — Credentials: YouTube (required) + GitHub push (optional)
# Best: store as Colab Secrets (key icon, left sidebar). Names:
#   YOUTUBE_REFRESH_TOKEN, YOUTUBE_CLIENT_ID, YOUTUBE_CLIENT_SECRET, GITHUB_TOKEN
# Otherwise you'll be prompted (hidden input; leave GITHUB_TOKEN blank to skip auto-push).
import os
from getpass import getpass
def _secret(name):
    try:
        from google.colab import userdata
        v = userdata.get(name)
        if v: return v
    except Exception:
        pass
    return getpass(f'{name} (blank to skip): ').strip()

# YouTube creds -> repo .env (gitignored)
yt = {k: _secret(k) for k in ('YOUTUBE_REFRESH_TOKEN','YOUTUBE_CLIENT_ID','YOUTUBE_CLIENT_SECRET')}
with open(os.path.join(REPO, '.env'), 'w') as f:
    for k, v in yt.items():
        f.write(f'{k}={v}\n')

# GitHub token -> auto commit+push of the site (optional)
gh = _secret('GITHUB_TOKEN')
PUSH = bool(gh)
if PUSH:
    os.environ['GITHUB_TOKEN'] = gh
    !git -C "$REPO" config user.email 'kutlwanomelamu93@gmail.com'
    !git -C "$REPO" config user.name  '100 Days of AI (Colab)'
    print('Auto-push ENABLED — the live site will update after each render.')
else:
    print('Auto-push OFF — videos.json will be saved to Drive for you to copy back.')

## Section 2 — Render + publish

Run the function cell, then call `render_and_publish(day, lesson)` for one lesson or loop a day.
Each call: **prep → Wav2Lip → finalize → YouTube upload → (if a GitHub token was given) commit + push**.
With auto-push on, that's the whole loop — nothing to do on your machine.

In [ ]:
# Cell 6 — the pipeline function
import os, glob, shutil, subprocess

def _sh(cmd, cwd):
    print('$', cmd)
    if subprocess.run(cmd, shell=True, cwd=cwd).returncode != 0:
        raise RuntimeError(f'FAILED: {cmd}')

def render_and_publish(day, lesson):
    lid = f'day_{day:03d}_lesson_{lesson:02d}'
    ym = glob.glob(f'{REPO}/*/day_{day:03d}/lessons/{lid}.yaml')
    if not ym:
        print(f'!! no YAML for {lid} — skipping'); return
    yaml = ym[0]
    print(f'\n========== {lid} ==========')
    # Stage 1 — slides + audio
    _sh(f'python 00_pipeline/lesson_build.py "{yaml}" --prep', REPO)
    audio = f'{REPO}/00_pipeline/audio/{lid}.mp3'
    assert os.path.exists(audio), 'prep produced no audio'
    # Stage 2 — talking head (write into repo so finalize finds it)
    th = f'{REPO}/00_pipeline/talking_heads/{lid}_talking_head.mp4'
    os.makedirs(os.path.dirname(th), exist_ok=True)
    _sh('python inference.py --checkpoint_path checkpoints/wav2lip_gan.pth '
        f'--face /content/avatar.png --audio "{audio}" --outfile "{th}" '
        '--pads 0 15 0 0 --fps 25 --nosmooth --resize_factor 1', '/content/Wav2Lip')
    assert os.path.exists(th), 'Wav2Lip produced no output'
    # Stage 3 — final composite
    _sh(f'python 00_pipeline/lesson_build.py "{yaml}" --finalize', REPO)
    final = f'{REPO}/00_pipeline/final/{lid}_final.mp4'
    assert os.path.exists(final), 'finalize produced no video'
    shutil.copy(final, f'{DRIVE}/final/{lid}_final.mp4')
    # Upload to YouTube; --push also commits + pushes the site if a GitHub token is set
    flag = ' --push' if PUSH else ''
    _sh(f'python tools/youtube_upload.py {lid}{flag}', REPO)
    shutil.copy(f'{REPO}/docs/videos.json', f'{DRIVE}/videos.json')
    print(f'========== {lid} DONE ==========')

print('render_and_publish() ready. PUSH =', PUSH)

In [ ]:
# Cell 7 — one lesson
render_and_publish(1, 1)

In [ ]:
# Cell 8 — a whole day (all 5 lessons)
for L in range(1, 6):
    render_and_publish(1, L)

## If auto-push is OFF

Without a GitHub token, video IDs are written to `docs/videos.json` inside the Colab repo and copied to **Drive: `MyDrive/100DaysOfAI/videos.json`**. To update the live site:

1. Download `videos.json` from Drive.
2. Replace `docs/videos.json` in your local repo.
3. `python tools/build_site_manifest.py`, then commit & push.

To enable auto-push instead: create a **fine-grained GitHub token** (this repo → *Contents: read and write*), add it as the Colab Secret `GITHUB_TOKEN`, and re-run Cell 5. Final MP4s are always archived to `MyDrive/100DaysOfAI/final/`.